In [22]:
# Warning control
import warnings
warnings.filterwarnings('ignore')

In [24]:
from crewai_tools import BaseTool
from crewai import Agent, Task, Crew
from pydantic import BaseModel, Field
from typing import List, Dict, ClassVar, Type
import requests
from bs4 import BeautifulSoup
from datetime import datetime
from dateutil import parser
import pytz
from langchain.tools import StructuredTool
import logging
from tenacity import retry, stop_after_attempt, wait_fixed
from httpx import RemoteProtocolError
from dotenv import load_dotenv
import os
from langchain.agents import AgentOutputParser
import json
import ast

In [25]:
# Load environment variables
load_dotenv()

openai_api_key = os.getenv("OPENAI_API_KEY")
os.environ["OPENAI_MODEL_NAME"] = 'gpt-4o'

In [26]:
# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# Define the input schema for the tool
class ScrapingToolInput(BaseModel):
    urls: List[str] = Field(
        ...,
        description="A list of URLs to scrape data from."
    )

# Define the output schema for the tool
class ScrapingToolOutput(BaseModel):
    data: List[Dict] = Field(
        ...,
        description="A list of dictionaries containing scraped data with title, subtitle, paragraphs, date, and link."
    )

# Define the custom scraping tool
class WebScrapingTool(BaseTool):
    name: str = "Web Scraping Tool"
    description: str = "Scrapes only today's news articles, extracting title, subtitle, and paragraphs."

    headers: ClassVar[dict] = {  # Annotate as ClassVar to avoid Pydantic validation error
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/109.0.0.0 Safari/537.36"
    }

    def _run(self, urls: List[str]) -> ScrapingToolOutput:
        """Scrape data from provided URLs, filtering only today's articles."""
        links = self._extract_links(urls)
        data = self._scrape_data(links)
        return ScrapingToolOutput(data=data)

    @retry(stop=stop_after_attempt(3), wait=wait_fixed(2), retry=lambda e: isinstance(e, RemoteProtocolError))
    def _extract_links(self, urls: List[str]) -> List[str]:
        """Extracts all article links from the provided URLs."""
        link_class = 'headline'  # Adjust this based on the actual website
        links = []
        for url in urls:
            try:
                response = requests.get(url, headers=self.headers, timeout=10)
                soup = BeautifulSoup(response.text, 'html.parser')
                sublist = [a.get('href') for a in soup.find_all('a', class_=link_class) if a.get('href')]
                links.extend(sublist)
            except Exception as e:
                 logging.error(f"Error fetching links from {url}: {str(e)}")
                 continue
        return links

    def _scrape_data(self, links: List[str]) -> List[Dict]:
        """Scrapes title, subtitle, and paragraphs from a list of links, filtering only today's articles."""
        data = []
        for link in links:
            try:
                article_data = self._scrape_article(link)
                if article_data and self._is_today(article_data['date']):
                    data.append(article_data)
            except Exception as e:
                logging.error(f"Error scraping {link}: {str(e)}")
                continue

        return data

    @retry(stop=stop_after_attempt(3), wait=wait_fixed(2), retry=lambda e: isinstance(e, RemoteProtocolError))
    def _scrape_article(self, link: str) -> Dict:
        """Scrapes a single article for title, subtitle, paragraphs, and date."""
        try:
            response = requests.get(link, headers=self.headers, timeout=10)
            soup = BeautifulSoup(response.text, 'html.parser')

            # Extract title using tag
            title = self._extract_element_using_tag(soup, 'h1')

            # Extract subtitle using tag
            subtitle = self._extract_element_using_tag(soup, 'h2')

            # Extract paragraphs using tag
            paragraphs = soup.find_all(['p', 'div'])

            main_paragraphs = [p.text.strip() for p in paragraphs]

            # Extract date using regex
            date_time = self._extract_element_using_regex(soup, 'time')

            return {
                'link': link,
                'title': title,
                'subtitle': subtitle,
                'paragraphs': main_paragraphs,
                'date': date_time
            }
        except Exception as e:
            logging.error(f"Error scraping article {link}: {str(e)}")
            return {
                'link': link,
                'title': None,
                'subtitle': None,
                'paragraphs': [],
                'date': None
            }

    def _extract_element_using_tag(self, soup: BeautifulSoup, tag: str) -> str:
        """Extracts the first matching element's text for a given tag"""
        element = soup.find(tag)
        if element:
            return element.text.strip()
        return None

    def _extract_element_using_regex(self, soup: BeautifulSoup, tag: str) -> str:
        """Extracts the first matching element's text for a given tag and class list using regex."""
        try:
            element = soup.find(tag)
            if element:
                # If the tag is <time>, extract from datetime attribute
                if tag == "time" and element.has_attr("datetime"):
                    return element["datetime"]  # Extracts ISO date format from <time>
                return element.text.strip()
        except Exception as e:
            logging.error(f"Error extracting element using regex: {str(e)}")
            return None

    def _is_today(self, date_string: str) -> bool:
        """Checks if the article date is today's date in the UK timezone."""
        
        if not date_string:
            logging.info("Skipping article with no date")
            return False

        try:
            # Define UK timezone
            uk_tz = pytz.timezone("Europe/London")
            
            # Print extracted date for debugging
            # print(f"Extracted date: {date_string}")  
            
            # Try parsing ISO format (e.g., "2024-01-21T14:32:00Z")
            if "T" in date_string:
                article_date = datetime.fromisoformat(date_string.split("T")[0])
            else:
                # Handle other date formats like "21 Jan"
                article_date = parser.parse(date_string, fuzzy=True)

            # Convert article date to UK time (if it isn't already localized)
            if article_date.tzinfo is None:
                article_date = pytz.utc.localize(article_date).astimezone(uk_tz)
            else:
                article_date = article_date.astimezone(uk_tz)

            # Get today's date in UK time
            today_uk = datetime.now(uk_tz).date()

            # Compare article date with today's date
            return article_date.date() == today_uk

        except Exception as e:
            logging.error(f"Date parsing issue: {date_string}, Error: {e}")
            return False

In [27]:
class CustomScrapeTool(StructuredTool):
      def _run(self, urls: List[str]) -> ScrapingToolOutput:
           return WebScrapingTool()._run(urls=urls)
      name: str = "Custom Web Scraping Tool"
      description: str = (
        "Scrapes crime-related news articles from provided URLs. Extracts titles, subtitles, and paragraphs."
      )
      args_schema: Type[str] = "ScrapingToolInput"
      return_schema: Type[str] = "ScrapingToolOutput"

# Initialize tools
custom_scrape_tool = CustomScrapeTool()

In [28]:
# Create Agents
scraper_agent = Agent(
    role="Web Scraper",
    goal="Extract all relevant articles from given URLs.",
    backstory="Expert web content extractor. Known for thorough and comprehensive extractions. Focuses on article content.",
    tools=[custom_scrape_tool],
    verbose=True
)

analyst_agent = Agent(
    role="Crime Analyst",
    goal="Analyze articles and select the most shocking crimes. Focus on the most significant crimes.",
    backstory="Expert crime analyst with focus on crime severity and shock factor.",
    verbose=True
)

storyteller_agent = Agent(
    role="Crime Storyteller",
     goal="Rewrite selected crime articles into captivating stories and structure in the correct JSON format.",
    backstory="Expert storyteller who crafts engaging stories from raw articles. Meticulous about formatting.",
    verbose=True,
)

qa_agent = Agent(
    role="Quality Assurance Agent",
    goal="Check if the output is correct and follows the given format.",
    backstory="A meticulous QA agent that is very strict about format. Ensures all JSON is correct.",
    verbose=True
)

In [29]:
# Create Tasks
scrape_task = Task(
    description="Scrape all relevant articles from the following URLs {urls}",
    expected_output="All the article content, including title, subtitle, and paragraphs.",
    agent=scraper_agent
)

analyze_task = Task(
    description="Analyze the scraped articles and choose the most shocking crime from each website. Provide context about the crime.",
    expected_output="List of most shocking crimes with a brief context.",
    agent=analyst_agent,
    context=[scrape_task]
)

class JSONOutputParser(AgentOutputParser):
    def parse(self, text: str) ->  dict:
        cleaned_output = text.strip()
        try:
            if cleaned_output.startswith("```json"):
                cleaned_output = cleaned_output[len("```json"):]
            if cleaned_output.endswith("```"):
                cleaned_output = cleaned_output[: -len("```")]
            # Attempt to parse JSON, but fallback to a dict-like structure if necessary
            try:
                return json.loads(cleaned_output)
            except json.JSONDecodeError:
                try:
                    #Attempt to parse a python-like dictionary
                    return ast.literal_eval(cleaned_output)
                except (ValueError, SyntaxError) as e:
                    logging.error(f"Error parsing JSON or dict-like string: {e}, the original text was {cleaned_output}")
                    return {
                        "data": []
                    }

        except Exception as e:
            logging.error(f"Error parsing JSON: {e} the original text was {text}")
            return {
            "data": []
            }

story_task = Task(
    description="Rewrite the selected shocking crimes into a story-telling and captivating way, and make sure that the output is always in json format, include the source link for each article. The keys should be: 'title', 'subtitle', 'story_body', and 'source_url'.",
     expected_output="A JSON formatted list of the rewritten shocking crime stories, each including the title, subtitle, story body, and source URL.",
    agent=storyteller_agent,
    context=[analyze_task],
    output_parser=JSONOutputParser()
)

qa_task = Task(
    description="Verify that the output from the story teller is in the correct JSON format and all fields are present.The keys should be: 'title', 'subtitle', 'story_body', and 'source_url'.",
    expected_output="Output confirmation if the output is in correct json format.",
    agent=qa_agent,
    context=[story_task]
)

In [30]:
# Create Crew
crime_crew = Crew(
    agents=[scraper_agent, analyst_agent, storyteller_agent, qa_agent],
    tasks=[scrape_task, analyze_task, story_task, qa_task],
    verbose=True
)

2025-01-29 16:53:58,511 - 5212 - __init__.py-__init__:537 - WARNING: Overriding of current TracerProvider is not allowed


In [31]:
# Run crew
crime_urls = {
    "urls": [
        "https://www.mylondon.news/all-about/crime"
        # "https://www.birminghammail.co.uk/all-about/crime",
        # "https://www.manchestereveningnews.co.uk/all-about/crime",
        # "https://www.liverpoolecho.co.uk/all-about/crime",
        # "https://www.walesonline.co.uk/all-about/crime"
    ]
}
result = crime_crew.kickoff(inputs=crime_urls)

 [DEBUG]: == Working Agent: Web Scraper
 [INFO]: == Starting Task: Scrape all relevant articles from the following URLs ['https://www.mylondon.news/all-about/crime']


> Entering new CrewAgentExecutor chain...


APIError: The model produced invalid content. Consider modifying your prompt if you are seeing this error persistently.

In [ ]:
print(result)